<a href="https://colab.research.google.com/github/dennisgathu8/36CHAMBERS/blob/main/ARIMA(Netflix).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
from google.colab import files

print("Please upload the file")
uploaded1 = files.upload()
file1_name = list(uploaded1.keys())[0]
df = pd.read_csv(file1_name)
df['Time Period'] = pd.to_datetime(df['Time Period'], format='%d/%m/%Y')
df.set_index('Time Period', inplace=True)
df = df.sort_index()
df.head()

Please upload the file


Saving Netflix-Subscriptions.csv to Netflix-Subscriptions.csv


,Subscribers
Time Period,
2013-04-01,34240000
2013-07-01,35640000
2013-10-01,38010000
2014-01-01,41430000
2014-04-01,46130000


In [3]:
import plotly.express as px
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df["Subscribers"], mode='lines', name='Subscribers'))
fig.update_layout(
    title="Netflix Subscribers Over Time",
    xaxis_title="Year",
    yaxis_title="Subscribers",
    template="plotly_white",
    width=900,
    height=500
)
fig.show()

In [4]:
!pip install statsmodels
from statsmodels.tsa.stattools import adfuller

def check_stationarity(timeseries):
  roll_mean = timeseries.rolling(window=12).mean()
  roll_std = timeseries.rolling(window=12).std()
  fig = go.Figure()
  fig.add_trace(go.Scatter(x=timeseries.index, y=timeseries,
                           mode='lines', name='Original',
                           line=dict(color='blue')))
  fig.add_trace(go.Scatter(x=roll_mean.index, y=roll_mean,
                           mode='lines', name='Rolling Mean',
                           line=dict(color='red', dash='dash')))
  fig.add_trace(go.Scatter(x=roll_std.index, y=roll_std,
                           mode='lines', name='Rolling Std',
                           line=dict(color='green', dash='dot')))
  fig.update_layout(
      title="Stationarity Check - Rolling Mean and Std",
      xaxis_title="Time",
      yaxis_title="Value",
      template="plotly_white",
      width=900,
      height=500
  )
  fig.show()
check_stationarity(df['Subscribers'])

In [6]:
df['Subscribers_diff'] = df['Subscribers'].diff().dropna()

from statsmodels.tsa.stattools import acf, pacf
def plot_acf_pacf(series, lags=20):
  series = series.dropna()
  acf_values = acf(series, nlags=lags)
  pacf_values = pacf(series, nlags=lags)
  fig_acf = go.Figure()
  fig_acf.add_trace(go.Bar(x=list(range(len(acf_values))), y=acf_values, name='ACF'))
  fig_acf.update_layout(title="Autocorrelation Function (ACF)", xaxis_title="ACF Value", template="plotly_white")

  fig_pacf = go.Figure()
  fig_pacf.add_trace(go.Bar(x=list(range(len(pacf_values))), y=pacf_values, name='PACF'))
  fig_pacf.update_layout(title="Partial Autocorrelation Function (PACF)", xaxis_title="Lags", yaxis_title="PACF Value", template="plotly_white")
  fig_acf.show()
  fig_pacf.show()
plot_acf_pacf(df['Subscribers_diff'], lags=20)

In [7]:
from statsmodels.tsa.arima.model import ARIMA
model = ARIMA(df['Subscribers'], order=(1,1,1))
model_fit = model.fit()
print(model_fit.summary())

/usr/local/lib/python3.11/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

No frequency information was provided, so inferred frequency QS-OCT will be used.



                               SARIMAX Results                                
Dep. Variable:            Subscribers   No. Observations:                   42
Model:                 ARIMA(1, 1, 1)   Log Likelihood                -672.993
Date:                Fri, 31 Jan 2025   AIC                           1351.986
Time:                        09:07:48   BIC                           1357.127
Sample:                    04-01-2013   HQIC                          1353.858
                         - 07-01-2023                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.9997      0.012     80.765      0.000       0.975       1.024
ma.L1         -0.9908      0.221     -4.476      0.000      -1.425      -0.557
sigma2      1.187e+13   1.57e-14   7.57e+26      0.0

In [10]:
def plot_forecast(df, forecast, model_fit, steps=12, freq='Q'):
  future_dates = pd.date_range(start=df.index[-1], periods=steps + 1, freq=freq)[1:]
  fig = go.Figure()
  fig.add_trace(go.Scatter(
      x=df.index, y=df['Subscribers'],
      mode='lines', name='Actual',
      line=dict(color='blue')
  ))
  fig.add_trace(go.Scatter(
      x=future_dates, y=forecast,
      mode='lines', name='Forecast',
      line=dict(color='red', dash='dash')
  ))
  fig.update_layout(
      title="Netflix Subscribers Forecast",
      xaxis_title="Time",
      yaxis_title="Subscribers",
      template="plotly_white",
      width=900,
      height=500
  )
  fig.show()
forecast = model_fit.forecast(steps=12)
plot_forecast(df, forecast, model_fit, steps=12, freq='Q')

<ipython-input-10-2b9bfd7f1fce>:2: FutureWarning:

'Q' is deprecated and will be removed in a future version, please use 'QE' instead.

